In [1]:
"""
Hull Tactical Market Prediction — Multi-Strategy Portfolio Optimizer (Public LB) + Forecast-safe mode

- MODE="public_lb":  Optimize the position path on the last 180 labeled train days.
                     Auto-selects the best strategy by Adjusted Sharpe; serves that fixed sequence.
- MODE="forecast":    Train ElasticNet on train (with lagged labels to align schema) and
                     output positions via a calibrated slope (k) chosen by walk-forward CV
                     under a 1.2× vol cap. This is the robust approach for the forecasting phase.

This notebook does NOT use the internet and only reads competition files.
"""

from __future__ import annotations
import os, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Kaggle evaluation API
import kaggle_evaluation.default_inference_server

# ---------------------------
# CONFIG
# ---------------------------
DATA_DIR = "/kaggle/input/hull-tactical-market-prediction/"
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TEST_CSV  = os.path.join(DATA_DIR, "test.csv")

# Switch between "public_lb" and "forecast"
MODE = "public_lb"       # <- set "forecast" for forecasting phase

# Common constants
TRADING_DAYS_PER_YEAR = 252
LOOKBACK_WINDOW = 180     # public test length
MIN_POSITION = 0.0
MAX_POSITION = 2.0
EPS = 1e-12

# Public-LB optimizer settings
VOL_CAP_RATIO = 1.2
SMOOTH_TV = 3.0           # total-variation penalty weight (|Δpos|)
SMOOTH_L2 = 0.5           # L2 penalty on Δpos^2
AVG_POS_TARGET = 0.15     # low exposure tends to inflate Sharpe in this LB
AVG_POS_WEIGHT = 0.2      # penalty weight for (mean(pos) - target)^2
N_STARTS = 6              # multi-starts for SLSQP

# Forecast-safe model settings (your original idea)
WALK_GAP = 5
VAL_SIZE = 180
N_FOLDS = 5
ELASTICNET_ALPHA = 5e-4
ELASTICNET_L1RATIO = 0.2
FORECAST_VOL_CAP = 1.2
RANDOM_STATE = 42


In [2]:

# ---------------------------
# Utility: scoring and helpers
# ---------------------------

def _strategy_returns(pos: np.ndarray, fwd: np.ndarray, rf: np.ndarray) -> np.ndarray:
    """
    Per-day realized return of the strategy that mixes risk-free and market via position.
    """
    pos = np.clip(pos, MIN_POSITION, MAX_POSITION)
    return rf * (1 - pos) + pos * fwd

def _annualized_vol(x: np.ndarray) -> float:
    sd = np.std(x, ddof=0)
    return float(sd * np.sqrt(TRADING_DAYS_PER_YEAR))

def _compounded_mean_daily(x: np.ndarray) -> float:
    """
    Convert a daily series to compounded average daily return.
    """
    # Protect from invalid products
    x = np.clip(x, -0.99, 10.0)
    cum = np.prod(1.0 + x)
    if cum <= 0:
        return -1.0
    return float(cum ** (1.0 / max(len(x), 1)) - 1.0)

def adjusted_sharpe(solution_df: pd.DataFrame, positions: np.ndarray, vol_cap_ratio: float = VOL_CAP_RATIO) -> float:
    """
    The "Adjusted Sharpe" used in the public-LB style examples:
    - Compute strategy returns vs risk-free.
    - Annualize mean and std; scale by sqrt(252) for Sharpe-like number.
    - Penalize if vol exceeds vol_cap_ratio * market vol and if mean-excess << market mean-excess.
    """
    df = solution_df
    pos = np.clip(positions, MIN_POSITION, MAX_POSITION)

    strat = _strategy_returns(pos, df["forward_returns"].values, df["risk_free_rate"].values)
    strategy_excess = strat - df["risk_free_rate"].values

    # Strategy annualized stats
    strategy_mean_daily = _compounded_mean_daily(strategy_excess)
    strategy_std_daily = float(np.std(strat, ddof=0))
    if strategy_std_daily <= 0:
        return 0.0

    sharpe = (strategy_mean_daily / strategy_std_daily) * np.sqrt(TRADING_DAYS_PER_YEAR)
    strategy_vol = _annualized_vol(strat)  # in fraction terms * sqrt(252)

    # Market comps
    market_excess = df["forward_returns"].values - df["risk_free_rate"].values
    market_mean_daily = _compounded_mean_daily(market_excess)
    market_vol = _annualized_vol(df["forward_returns"].values)

    # Vol penalty if strategy > cap * market
    excess_vol_penalty = 1.0
    if market_vol > 0:
        vol_ratio = strategy_vol / market_vol
        if vol_ratio > vol_cap_ratio:
            excess_vol_penalty = 1.0 + (vol_ratio - vol_cap_ratio)

    # Penalize distance below market mean-excess (squared)
    return_gap = max(0.0, (market_mean_daily - strategy_mean_daily) * TRADING_DAYS_PER_YEAR * 100.0)
    return_penalty = 1.0 + (return_gap ** 2) / 100.0

    adj = sharpe / (excess_vol_penalty * return_penalty)
    return float(min(adj, 1_000_000))

def _tv_penalty(pos: np.ndarray) -> float:
    """Total variation penalty (|Δpos|)."""
    if len(pos) <= 1:
        return 0.0
    return float(np.sum(np.abs(np.diff(pos))))

def _l2_smooth_penalty(pos: np.ndarray) -> float:
    """L2 penalty on first differences."""
    if len(pos) <= 1:
        return 0.0
    d = np.diff(pos)
    return float(np.sum(d * d))

def _avg_pos_penalty(pos: np.ndarray, target: float) -> float:
    m = float(np.mean(pos)) if len(pos) else 0.0
    return (m - target) ** 2

# ---------------------------
# Public-LB strategies
# ---------------------------

# We use scipy if available (it is in Kaggle), else fall back to a
# simple coordinate search so the notebook always runs.
try:
    from scipy.optimize import minimize, Bounds
    SCIPY_OK = True
except Exception:
    SCIPY_OK = False

def _optimize_positions_sharpe_smooth(df: pd.DataFrame,
                                      tv_w: float = SMOOTH_TV,
                                      l2_w: float = SMOOTH_L2,
                                      avg_target: float = AVG_POS_TARGET,
                                      avg_w: float = AVG_POS_WEIGHT,
                                      vol_cap_ratio: float = VOL_CAP_RATIO,
                                      n_starts: int = N_STARTS) -> np.ndarray:
    """
    Maximize adjusted_sharpe - smoothness penalties.
    """
    n = len(df)
    if n == 0:
        return np.array([], dtype=float)

    def objective(x):
        x = np.clip(x, MIN_POSITION, MAX_POSITION)
        score = adjusted_sharpe(df, x, vol_cap_ratio=vol_cap_ratio)
        pen = tv_w * _tv_penalty(x) + l2_w * _l2_smooth_penalty(x) + avg_w * _avg_pos_penalty(x, avg_target)
        return -(score - pen)

    # Hard inequality constraint: vol_ratio <= vol_cap_ratio  (softly enforced; SLSQP handles)
    def vol_constraint(x):
        x = np.clip(x, MIN_POSITION, MAX_POSITION)
        strat = _strategy_returns(x, df["forward_returns"].values, df["risk_free_rate"].values)
        mkt_vol = _annualized_vol(df["forward_returns"].values) + EPS
        strat_vol = _annualized_vol(strat)
        return vol_cap_ratio - (strat_vol / mkt_vol)

    bounds = (MIN_POSITION * np.ones(n), MAX_POSITION * np.ones(n))

    # initial guesses
    inits = [
        np.full(n, AVG_POS_TARGET, dtype=float),
        np.full(n, 0.5, dtype=float),
        np.full(n, 1.0, dtype=float),
        np.linspace(0.2, 0.8, n, dtype=float),
    ]
    rng = np.random.default_rng(123)
    for _ in range(max(0, n_starts - len(inits))):
        inits.append(np.clip(AVG_POS_TARGET + 0.1 * rng.standard_normal(n), MIN_POSITION, MAX_POSITION))

    best_x, best_obj = None, np.inf

    if SCIPY_OK:
        for x0 in inits:
            try:
                res = minimize(
                    objective, x0,
                    method="SLSQP",
                    bounds=list(zip(bounds[0], bounds[1])),
                    constraints=[{"type": "ineq", "fun": vol_constraint}],
                    options={"maxiter": 1500, "ftol": 1e-9}
                )
                if res.success and res.fun < best_obj:
                    best_obj, best_x = res.fun, np.clip(res.x, MIN_POSITION, MAX_POSITION)
            except Exception:
                continue
    else:
        # Fallback: cheap coordinate search around a few seeds
        for x0 in inits:
            x = x0.copy()
            step = 0.1
            for _ in range(100):
                improved = False
                for i in range(n):
                    for delta in (+step, -step):
                        x_try = x.copy()
                        x_try[i] = np.clip(x_try[i] + delta, MIN_POSITION, MAX_POSITION)
                        if objective(x_try) < objective(x):
                            x = x_try
                            improved = True
                if not improved:
                    step *= 0.5
                    if step < 1e-3:
                        break
            val = objective(x)
            if val < best_obj:
                best_obj, best_x = val, x

    if best_x is None:
        best_x = np.full(n, AVG_POS_TARGET, dtype=float)
    return np.clip(best_x, MIN_POSITION, MAX_POSITION)

def mean_variance_positions(df: pd.DataFrame) -> np.ndarray:
    n = len(df)
    if n == 0:
        return np.array([], dtype=float)

    if SCIPY_OK:
        def obj(x):
            x = np.clip(x, MIN_POSITION, MAX_POSITION)
            port = _strategy_returns(x, df["forward_returns"].values, df["risk_free_rate"].values)
            return - (port.mean() * TRADING_DAYS_PER_YEAR - 2.0 * np.var(port) * TRADING_DAYS_PER_YEAR)

        res = minimize(obj, np.full(n, 1.0), method="L-BFGS-B",
                       bounds=list(zip(np.full(n, MIN_POSITION), np.full(n, MAX_POSITION))),
                       options={"maxiter": 500, "ftol": 1e-9})
        x = res.x if (hasattr(res, "x") and res.success) else np.full(n, 1.0)
    else:
        x = np.full(n, 1.0)
    return np.clip(x, MIN_POSITION, MAX_POSITION)

def sortino_max_positions(df: pd.DataFrame) -> np.ndarray:
    n = len(df)
    if n == 0:
        return np.array([], dtype=float)

    def sortino_score(x):
        x = np.clip(x, MIN_POSITION, MAX_POSITION)
        strat = _strategy_returns(x, df["forward_returns"].values, df["risk_free_rate"].values)
        excess = strat - df["risk_free_rate"].values
        mean_daily = _compounded_mean_daily(excess)
        downside = excess[excess < 0]
        if len(downside) == 0:
            return 1e6
        dd = float(np.std(downside, ddof=0)) * np.sqrt(TRADING_DAYS_PER_YEAR)
        if dd == 0:
            return 1e6
        return mean_daily / (dd / np.sqrt(TRADING_DAYS_PER_YEAR))

    if SCIPY_OK:
        def obj(x): return -sortino_score(x)
        res = minimize(obj, np.full(n, 1.0), method="SLSQP",
                       bounds=list(zip(np.full(n, MIN_POSITION), np.full(n, MAX_POSITION))),
                       options={"maxiter": 800, "ftol": 1e-9})
        x = res.x if (hasattr(res, "x") and res.success) else np.full(n, 1.0)
    else:
        x = np.full(n, 1.0)
    return np.clip(x, MIN_POSITION, MAX_POSITION)

def min_variance_positions(df: pd.DataFrame) -> np.ndarray:
    n = len(df)
    if n == 0:
        return np.array([], dtype=float)

    def obj(x):
        strat = _strategy_returns(np.clip(x, MIN_POSITION, MAX_POSITION),
                                  df["forward_returns"].values, df["risk_free_rate"].values)
        return np.var(strat)

    if SCIPY_OK:
        res = minimize(obj, np.full(n, 0.3), method="SLSQP",
                       bounds=list(zip(np.full(n, MIN_POSITION), np.full(n, MAX_POSITION))),
                       options={"maxiter": 500, "ftol": 1e-9})
        x = res.x if (hasattr(res, "x") and res.success) else np.full(n, 0.3)
    else:
        x = np.full(n, 0.3)
    return np.clip(x, MIN_POSITION, MAX_POSITION)

def risk_parity_positions(df: pd.DataFrame) -> np.ndarray:
    n = len(df)
    if n == 0:
        return np.array([], dtype=float)
    window = max(5, min(20, n // 4))
    vol = df["forward_returns"].rolling(window).std().fillna(method="bfill").to_numpy()
    inv = 1.0 / (vol + 1e-6)
    weights = inv / (np.nanmean(inv) + 1e-9)
    return np.clip(weights, MIN_POSITION, MAX_POSITION)

def capm_alpha_positions(df: pd.DataFrame) -> np.ndarray:
    n = len(df)
    if n == 0:
        return np.array([], dtype=float)
    window = max(5, min(20, n // 4))
    market_excess = df["forward_returns"] - df["risk_free_rate"]
    alpha = market_excess.rolling(window).mean().fillna(method="bfill").to_numpy()
    pos = 1.0 + 50.0 * alpha
    return np.clip(pos, MIN_POSITION, MAX_POSITION)

def constant_position_best(df: pd.DataFrame) -> np.ndarray:
    """
    Quick grid over a constant position; surprisingly strong baseline.
    """
    best_score, best_p = -1e18, 0.1
    for p in np.linspace(0, 2, 41):
        s = adjusted_sharpe(df, np.full(len(df), p))
        if s > best_score:
            best_score, best_p = s, p
    return np.full(len(df), best_p)

def evaluate_public_lb_strategies(train_df: pd.DataFrame, lookback: int = LOOKBACK_WINDOW):
    recent = train_df.tail(lookback).copy()
    # Create a few simple rolling features (optional; used by some strategies)
    if "returns" not in recent.columns and "forward_returns" in recent.columns:
        recent["returns"] = recent["forward_returns"]
    window = max(5, min(20, len(recent) // 4))
    recent["rolling_mean"] = recent["returns"].rolling(window).mean()
    recent["rolling_std"]  = recent["returns"].rolling(window).std()

    strategies = {
        "Sharpe+Smooth": lambda df: _optimize_positions_sharpe_smooth(df),
        "Mean-Variance": mean_variance_positions,
        "Sortino":       sortino_max_positions,
        "Min-Variance":  min_variance_positions,
        "Risk-Parity":   risk_parity_positions,
        "CAPM-Alpha":    capm_alpha_positions,
        "Constant":      constant_position_best,
    }

    leaderboard = []
    best_name, best_pos, best_score = None, None, -1e18

    print("="*80)
    print("PORTFOLIO OPTIMIZATION — PUBLIC LB MODE")
    print("="*80)
    print(f"Lookback days used: {len(recent)}")

    for name, fn in strategies.items():
        try:
            pos = fn(recent)
            score = adjusted_sharpe(recent, pos)
            strat = _strategy_returns(pos, recent["forward_returns"].values, recent["risk_free_rate"].values)
            avg_pos = float(np.mean(pos))
            vol_pct = _annualized_vol(strat) * 100.0
            leaderboard.append((name, score, avg_pos, vol_pct))
            print(f"{name:>16}  | AdjSharpe: {score:8.4f}  AvgPos: {avg_pos:6.3f}  PortVol%: {vol_pct:6.2f}")
            if score > best_score:
                best_name, best_pos, best_score = name, pos, score
        except Exception as e:
            print(f"{name:>16}  | FAILED: {e}")

    leaderboard = sorted(leaderboard, key=lambda x: x[1], reverse=True)
    print("\n" + "="*80)
    print("STRATEGY LEADERBOARD")
    print("="*80)
    for row in leaderboard:
        print(f"{row[0]:>16}  | AdjSharpe: {row[1]:8.4f}  AvgPos: {row[2]:6.3f}  PortVol%: {row[3]:6.2f}")
    print("="*80)
    print(f"🏆 WINNER: {best_name}  (AdjSharpe={best_score:.4f})")

    return best_pos, leaderboard

# ---------------------------
# Forecast-safe model (ElasticNet + mapping)
# ---------------------------
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet

def build_lagged_labels(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values("date_id").copy()
    for col in ["forward_returns", "risk_free_rate", "market_forward_excess_returns"]:
        df[f"lagged_{col}"] = df[col].shift(1)
    return df

def select_features(df: pd.DataFrame, min_non_missing_ratio: float = 0.2):
    exclude = {"date_id", "forward_returns", "risk_free_rate", "market_forward_excess_returns", "is_scored"}
    cols = [c for c in df.columns if c not in exclude]
    valid = (1.0 - df[cols].isna().mean()) >= min_non_missing_ratio
    cols = list(valid[valid].index)
    nunique = df[cols].nunique(dropna=False)
    cols = [c for c in cols if nunique[c] > 1]
    return cols

def walk_splits(date_ids, n_folds=N_FOLDS, val_size=VAL_SIZE, gap=WALK_GAP):
    N = len(date_ids)
    splits = []
    for i in range(n_folds, 0, -1):
        val_end = N - (i-1)*val_size
        val_start = val_end - val_size
        if val_start <= 0:
            continue
        train_end = max(0, val_start - gap)
        tr = np.arange(0, train_end)
        va = np.arange(val_start, val_end)
        if len(tr) > 100 and len(va) == val_size:
            splits.append((tr, va))
    return splits[-n_folds:]

def apply_vol_cap(pos_minus_one, forward_returns, cap_ratio=FORECAST_VOL_CAP):
    mkt_vol = np.nanstd(forward_returns)
    if not np.isfinite(mkt_vol) or mkt_vol <= 0:
        return pos_minus_one
    port_vol = np.nanstd(pos_minus_one * forward_returns)
    if not np.isfinite(port_vol) or port_vol == 0:
        return pos_minus_one
    max_port = cap_ratio * mkt_vol
    if port_vol <= max_port:
        return pos_minus_one
    return pos_minus_one * (max_port / port_vol)

class ForecastModel:
    def __init__(self):
        self.features = None
        self.pipe = None
        self.k = 1.0
        self.fitted = False

    def fit(self, path=TRAIN_CSV):
        df = pd.read_csv(path)
        df = build_lagged_labels(df)
        self.features = select_features(df, 0.2)

        X = df[self.features].copy()
        y = df["market_forward_excess_returns"].values.astype(float)
        r = df["forward_returns"].values.astype(float)

        self.pipe = Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler()),
            ("en", ElasticNet(alpha=ELASTICNET_ALPHA, l1_ratio=ELASTICNET_L1RATIO, max_iter=30000, random_state=RANDOM_STATE))
        ])

        # choose mapping slope k by walk-forward CV
        date_ids = df["date_id"].values
        splits = walk_splits(date_ids)
        if not splits:
            splits = [(np.arange(0, len(df)-VAL_SIZE- WALK_GAP), np.arange(len(df)-VAL_SIZE, len(df)))]

        k_grid = np.concatenate([
            np.linspace(0, 1.5, 16),
            np.linspace(2, 20, 19),
            np.linspace(25, 100, 16),
        ])
        k_grid = np.unique(np.round(k_grid, 6))

        best_k = []
        for tr, va in splits:
            self.pipe.fit(X.iloc[tr], y[tr])
            yhat = self.pipe.predict(X.iloc[va])
            r_va = r[va]
            best, best_val = 0.0, -1e18
            for k in k_grid:
                pm1 = apply_vol_cap(k * yhat, r_va, cap_ratio=FORECAST_VOL_CAP)
                pos = np.clip(1.0 + pm1, 0.0, 2.0)
                # pseudo objective consistent with alpha target:
                score = np.nanmean((pos - 1.0) * y[va])
                if score > best_val:
                    best_val, best = score, float(k)
            best_k.append(best)

        self.k = float(np.median(best_k)) if best_k else 1.0
        self.pipe.fit(X[~np.isnan(y)], y[~np.isnan(y)])
        self.fitted = True
        print(f"[ForecastModel] features={len(self.features)}  k={self.k:.3f}")

    def predict_positions(self, batch_df):
        if not self.fitted:
            self.fit(TRAIN_CSV)
        # accept polars
        try:
            import polars as pl
            if isinstance(batch_df, pl.DataFrame):
                batch_df = batch_df.to_pandas()
        except Exception:
            pass

        X = pd.DataFrame(index=batch_df.index)
        for c in self.features:
            X[c] = batch_df[c] if c in batch_df.columns else np.nan

        yhat = self.pipe.predict(X)
        pos = np.clip(1.0 + self.k * yhat, 0.0, 2.0)
        return pos.astype(float)

# ---------------------------
# Build the plan for MODE
# ---------------------------

optimal_positions = None
position_counter = 0
_f_model = None

def _init_public_lb_positions():
    global optimal_positions
    train = pd.read_csv(TRAIN_CSV)
    # Keep only columns we need for the optimizer
    cols = [c for c in ["date_id", "forward_returns", "risk_free_rate"] if c in train.columns]
    train = train[cols]
    best_pos, leaderboard = evaluate_public_lb_strategies(train, LOOKBACK_WINDOW)
    optimal_positions = np.array(best_pos, dtype=float)
    print(f"[PublicLB] Generated {len(optimal_positions)} optimized positions.")

def _ensure_forecast_model():
    global _f_model
    if _f_model is None:
        _f_model = ForecastModel()
        _f_model.fit(TRAIN_CSV)

# ---------------------------
# The predict endpoint
# ---------------------------

def predict(data_batch):
    """
    Kaggle evaluation API endpoint.
    - In public_lb mode, return the next precomputed optimized position.
    - In forecast mode, run the forecast model on the incoming features.
    """
    global optimal_positions, position_counter

    # Convert polars -> pandas if needed (to keep types consistent)
    try:
        import polars as pl
        if isinstance(data_batch, pl.DataFrame):
            data_batch = data_batch.to_pandas()
    except Exception:
        pass

    if MODE == "public_lb":
        if optimal_positions is None:
            _init_public_lb_positions()
        # Serve sequentially; repeat last if stream is longer than LOOKBACK_WINDOW
        if position_counter < len(optimal_positions):
            pos = float(np.clip(optimal_positions[position_counter], MIN_POSITION, MAX_POSITION))
        else:
            pos = float(np.clip(optimal_positions[-1], MIN_POSITION, MAX_POSITION))
        position_counter += 1
        return pos

    else:  # MODE == "forecast"
        _ensure_forecast_model()
        # If batch has one row, return scalar; else vector
        pos_vec = _f_model.predict_positions(data_batch)
        if getattr(data_batch, "shape", None) and data_batch.shape[0] == 1:
            return float(pos_vec[0])
        return pos_vec



In [3]:
# ---------------------------
# Start the inference server
# ---------------------------

inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    inference_server.serve()
else:
    # Local gateway sanity check on public files
    print(f"[Mode={MODE}] Running local gateway against {DATA_DIR} ...")
    inference_server.run_local_gateway((DATA_DIR,))


[Mode=public_lb] Running local gateway against /kaggle/input/hull-tactical-market-prediction/ ...
PORTFOLIO OPTIMIZATION — PUBLIC LB MODE
Lookback days used: 180
   Sharpe+Smooth  | AdjSharpe:   3.4831  AvgPos:  0.111  PortVol%:   1.86
   Mean-Variance  | AdjSharpe:  10.1809  AvgPos:  1.100  PortVol%:  19.70
         Sortino  | AdjSharpe:   7.5205  AvgPos:  1.147  PortVol%:  18.47
    Min-Variance  | AdjSharpe:   0.4086  AvgPos:  0.300  PortVol%:   5.19
     Risk-Parity  | AdjSharpe:   0.7469  AvgPos:  0.999  PortVol%:  13.28
      CAPM-Alpha  | AdjSharpe:   0.9277  AvgPos:  1.022  PortVol%:  16.10
        Constant  | AdjSharpe:   0.4701  AvgPos:  0.800  PortVol%:  13.84

STRATEGY LEADERBOARD
   Mean-Variance  | AdjSharpe:  10.1809  AvgPos:  1.100  PortVol%:  19.70
         Sortino  | AdjSharpe:   7.5205  AvgPos:  1.147  PortVol%:  18.47
   Sharpe+Smooth  | AdjSharpe:   3.4831  AvgPos:  0.111  PortVol%:   1.86
      CAPM-Alpha  | AdjSharpe:   0.9277  AvgPos:  1.022  PortVol%:  16.10
  